# MaIA_SDCI Final Project

**Sistemas de Decision y Control Inteligente (SDCI)**  
**Universidad de los Andes, 2026**

Copyright (c) 2026 Leffer Trochez

---

Initializes the complete project using paths relative to the project root and launches the same ipywidgets interface in VS Code/Jupyter or Google Colab.


Run all cells in this notebook. The project automatically creates `results/` when needed. All source notebooks are loaded from paths relative to the project root.


In [ ]:
from pathlib import Path
import importlib.util
import json


PROJECT_FOLDER_NAMES = [
    "MaIA_SDCI_Final_Project_Python",
    "MaIA_SDCI_Final_Project_PYTHON",
]


def _is_project_root(path):
    path = Path(path)
    return (
        (path / "PROJECT_STRUCTURE.txt").is_file()
        and (path / "requirements.txt").is_file()
        and (path / "config").is_dir()
        and (path / "app").is_dir()
        and (path / "utilities").is_dir()
    )


def _locate_project_root():
    cwd = Path.cwd().resolve()

    # Current folder or one of its parents.
    for candidate in [cwd, *cwd.parents]:
        if _is_project_root(candidate):
            return candidate

    # Common local / Colab upload locations.
    direct_candidates = []
    for folder_name in PROJECT_FOLDER_NAMES:
        direct_candidates.extend([
            cwd / folder_name,
            Path("/content") / folder_name,
        ])

    for candidate in direct_candidates:
        if _is_project_root(candidate):
            return candidate.resolve()

    # Google Colab + Google Drive.
    try:
        import google.colab  # noqa: F401
        from google.colab import drive

        drive_root = Path("/content/drive")
        my_drive = drive_root / "MyDrive"

        if not my_drive.exists():
            drive.mount(
                str(drive_root),
                force_remount=False,
            )

        drive_candidates = []
        for folder_name in PROJECT_FOLDER_NAMES:
            drive_candidates.extend([
                my_drive / folder_name,
                *list(
                    my_drive.glob(
                        f"*/{folder_name}"
                    )
                ),
            ])

        for candidate in drive_candidates:
            if _is_project_root(candidate):
                return candidate.resolve()

    except Exception:
        pass

    raise FileNotFoundError(
        "Project root could not be located. "
        "Open VS Code in the complete project folder, "
        "or in Colab place the complete "
        "MaIA_SDCI_Final_Project_Python folder in /content "
        "or in the root of MyDrive."
    )


def _check_dependencies(project_root):
    required_modules = [
        "numpy",
        "scipy",
        "pandas",
        "matplotlib",
        "ipywidgets",
        "IPython",
    ]

    missing = [
        name
        for name in required_modules
        if importlib.util.find_spec(name) is None
    ]

    if missing:
        requirements_file = (
            Path(project_root)
            / "requirements.txt"
        )

        command = (
            "python -m pip install -r "
            f'"{requirements_file}"'
        )

        raise RuntimeError(
            "Missing Python dependencies: "
            + ", ".join(missing)
            + "\nInstall the project requirements first:\n"
            + command
        )


def _execute_notebook(path, namespace):
    """Execute code cells from one project notebook in a shared namespace."""

    path = Path(path)
    notebook = json.loads(
        path.read_text(encoding="utf-8")
    )

    for index, cell in enumerate(
        notebook.get("cells", []),
        start=1,
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(
            cell.get("source", [])
        )

        if not source.strip():
            continue

        compiled = compile(
            source,
            f"{path}::cell_{index}",
            "exec",
        )
        exec(compiled, namespace)


def startup_project(project_root=None):
    """Initialize the complete MaIA_SDCI Python project."""

    project_root = (
        Path(project_root).resolve()
        if project_root
        else _locate_project_root()
    )

    _check_dependencies(project_root)

    # PROJECT FOLDERS
    config_folder = project_root / "config"
    model_config_folder = config_folder / "model"
    controller_folder = config_folder / "controller"
    app_folder = project_root / "app"
    utilities_folder = project_root / "utilities"
    results_folder = project_root / "results"

    # VALIDATE SOURCE FOLDERS
    required_source_folders = [
        config_folder,
        model_config_folder,
        controller_folder,
        app_folder,
        utilities_folder,
    ]

    for folder in required_source_folders:
        if not folder.is_dir():
            raise FileNotFoundError(
                "Required project folder was not found: "
                f"{folder}"
            )

    # GENERATED RESULTS FOLDER
    results_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    # PROJECT INFORMATION
    PROJECT = {
        "name": "MaIA_SDCI",
        "root": str(project_root),
        "configFolder": str(config_folder),
        "modelConfigFolder": str(
            model_config_folder
        ),
        "controllerFolder": str(
            controller_folder
        ),
        "appFolder": str(app_folder),
        "utilitiesFolder": str(
            utilities_folder
        ),
        "resultsFolder": str(
            results_folder
        ),
    }

    namespace = globals()
    namespace["PROJECT"] = PROJECT

    # LOAD PROJECT NOTEBOOKS
    notebook_order = [
        model_config_folder
        / "actuator_parameters.ipynb",
        model_config_folder
        / "plant_parameters.ipynb",
        config_folder
        / "project_config.ipynb",
        config_folder
        / "model_parameters.ipynb",
        config_folder
        / "environment_parameters.ipynb",
        config_folder
        / "sensor_parameters.ipynb",

        # All controller code is isolated here.
        controller_folder
        / "on_off_parameters.ipynb",
        controller_folder
        / "fuzzy_parameters.ipynb",
        controller_folder
        / "mpc_parameters.ipynb",
        controller_folder
        / "esc_parameters.ipynb",
        controller_folder
        / "replicator_parameters.ipynb",
        controller_folder
        / "controller_parameters.ipynb",

        utilities_folder
        / "compute_metrics.ipynb",
        utilities_folder
        / "save_results.ipynb",
        project_root
        / "MaIA_SDCI_Temperature_Control.ipynb",
        app_folder
        / "configure_logging.ipynb",
        app_folder
        / "Control_App.ipynb",
        app_folder
        / "launch_app.ipynb",
    ]

    for notebook_path in notebook_order:
        if not notebook_path.is_file():
            raise FileNotFoundError(
                "Required project notebook was not found: "
                f"{notebook_path}"
            )

        _execute_notebook(
            notebook_path,
            namespace,
        )

    # CONFIGURATION STRUCTURES
    CFG = project_config()
    MODEL = model_parameters()
    ENV = environment_parameters()
    SENSOR = sensor_parameters()

    (
        CTRL,
        CTRL_REPLICATOR,
        MPC_PLANT,
        MPC_CONTROLLER,
    ) = controller_parameters(MODEL)

    context = {
        "PROJECT": PROJECT,
        "CFG": CFG,
        "MODEL": MODEL,
        "ENV": ENV,
        "SENSOR": SENSOR,
        "CTRL": CTRL,
        "CTRL_REPLICATOR": CTRL_REPLICATOR,
        "MPC_PLANT": MPC_PLANT,
        "MPC_CONTROLLER": MPC_CONTROLLER,
        "CONTROLLER_ID": 1,
    }

    namespace["PROJECT_CONTEXT"] = context

    print("Project environment ready.")
    return context


In [ ]:
# Run this cell after the initialization cell above.
PROJECT_CONTEXT = startup_project()
APP = launch_app(PROJECT_CONTEXT)


Project configuration loaded.
  Stop time:   1800.0 s
  References:  [22.0 21.0 20.0 19.0] degC
  ESC ref.:    22.0 degC
Model parameters loaded.
  Equation-based actuator + plant
Environment parameters loaded.
  Ambient temperature:     8.0 degC
  Disturbance enabled:     0
  Disturbance amplitude:   13.0 degC
  Disturbance start time:  5.0 min
  Disturbance end time:    8.0 min
  Disturbance room mask:   [1 1 1 1]
Sensor parameters loaded.
  Time constant: 10.00 s
  Sample time:   1.00 s
  Noise std:     0.050 degC
Controller configuration loaded.
  Student workspace: config/controller/
  Interface: T_ref, T_error, T_meas -> duty_cycle
Project environment ready.


C:\Users\leffe\OneDrive - Universidad de los andes\PhD\Uniandes\MaIA\SDCI\Proyecto Final\MaIA_SDCI_Final_Project_Python\app\Control_App.ipynb::cell_2:834: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
